# Pertemuan 5 — Tugas 1: K-Means pada Mall Customers

Tujuan tugas ini adalah mengelompokkan pelanggan berdasarkan pendapatan tahunan dan skor belanja. Dataset `Mall_Customers.csv` diunduh dari lampiran [Tugas Praktikum](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js04-klasterisasi/tugas-praktikum).

Kita memilih dua fitur tersebut karena dapat menggambarkan kemampuan dan kebiasaan belanja pelanggan.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('Pertemuan 5/Mall_Customers.csv')
print('Ukuran data:', df.shape)
print('Jumlah nilai kosong:', df.isna().sum().sum())
print('Jumlah baris duplikat:', df.duplicated().sum())
df.head()


Ukuran data: (200, 5)
Jumlah nilai kosong: 0
Jumlah baris duplikat: 0


## 1. Memilih dua fitur dan menstandarkan data
`CustomerID` hanya identitas sehingga tidak digunakan. Kita memilih `Annual Income (k$)` dan `Spending Score (1-100)`. Pendapatan dan skor belanja mempunyai satuan berbeda, sehingga kita memakai StandardScaler sebelum menghitung jarak.


In [3]:
fitur = ['Annual Income (k$)', 'Spending Score (1-100)']
X_asli = df[fitur]
scaler = StandardScaler()
X = scaler.fit_transform(X_asli)
plt.figure(figsize=(7, 5))
plt.scatter(X_asli.iloc[:, 0], X_asli.iloc[:, 1], s=35)
plt.xlabel('Pendapatan tahunan (ribu dolar)')
plt.ylabel('Skor belanja')
plt.title('Sebaran Pelanggan Sebelum Klasterisasi')
plt.show()


## 2. Membandingkan jumlah klaster
Kita mencoba k = 1 sampai 10 untuk Elbow. Silhouette hanya dihitung mulai k = 2 karena memerlukan minimal dua klaster. SSE yang lebih kecil belum tentu menjadi pilihan terbaik, sehingga hasilnya dibandingkan dengan Silhouette.


In [4]:
hasil = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = model.fit_predict(X)
    sil = silhouette_score(X, labels) if k > 1 else np.nan
    hasil.append({'k': k, 'SSE': model.inertia_, 'Silhouette': sil})
evaluasi = pd.DataFrame(hasil)
display(evaluasi.round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(evaluasi['k'], evaluasi['SSE'], 'o-')
axes[0].set(xlabel='k', ylabel='SSE', title='Metode Elbow', xticks=range(1, 11))
axes[1].plot(evaluasi['k'], evaluasi['Silhouette'], 'o-')
axes[1].set(xlabel='k', ylabel='Silhouette', title='Perbandingan Silhouette', xticks=range(1, 11))
plt.tight_layout()
plt.show()


## 3. Memilih k terbaik dan membuat model akhir
Model akhir memakai k dengan Silhouette tertinggi di antara k = 2 sampai 10. Grafik Elbow dipakai sebagai pembanding untuk melihat apakah pilihan itu berada di daerah grafik yang mulai melandai. Pilihan ini berlaku untuk dua fitur dan pengolahan data yang digunakan di tugas ini.


In [5]:
baris_terbaik = evaluasi.loc[evaluasi['Silhouette'].idxmax()]
k_terbaik = int(baris_terbaik['k'])
print('k terpilih:', k_terbaik)
print(f"Silhouette terbaik: {baris_terbaik['Silhouette']:.4f}")
model_akhir = KMeans(n_clusters=k_terbaik, random_state=0, n_init=10)
df['Klaster'] = model_akhir.fit_predict(X)
centers = scaler.inverse_transform(model_akhir.cluster_centers_)
plt.figure(figsize=(9, 6))
for cluster in range(k_terbaik):
    anggota = df[df['Klaster'] == cluster]
    plt.scatter(anggota[fitur[0]], anggota[fitur[1]], s=45, label=f'Klaster {cluster}')
plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, marker='X', label='Centroid')
plt.xlabel('Pendapatan tahunan (ribu dolar)')
plt.ylabel('Skor belanja')
plt.title(f'Segmentasi Pelanggan dengan k = {k_terbaik}')
plt.legend()
plt.show()


k terpilih: 5
Silhouette terbaik: 0.5547


## 4. Menjelaskan kelompok pelanggan
Kita melihat jumlah anggota dan rata-rata dua fitur untuk memahami setiap kelompok. Nomor klaster hanya penanda dan tidak menunjukkan peringkat pelanggan.

Istilah rendah, sedang, dan tinggi di bawah dibandingkan dengan kuartil fitur pada dataset ini, bukan batas pendapatan masyarakat secara umum.


In [6]:
ringkasan = df.groupby('Klaster').agg(
    Jumlah_pelanggan=('CustomerID', 'count'),
    Rata_pendapatan=(fitur[0], 'mean'),
    Rata_skor_belanja=(fitur[1], 'mean'))
display(ringkasan.round(2))

def kategori(nilai, kolom):
    batas_bawah, batas_atas = df[kolom].quantile([0.25, 0.75])
    if nilai < batas_bawah:
        return 'rendah'
    if nilai > batas_atas:
        return 'tinggi'
    return 'sedang'

for cluster, row in ringkasan.iterrows():
    pendapatan = kategori(row['Rata_pendapatan'], fitur[0])
    belanja = kategori(row['Rata_skor_belanja'], fitur[1])
    print(f'Klaster {cluster}: {int(row["Jumlah_pelanggan"])} pelanggan, '
          f'pendapatan {pendapatan}, dan skor belanja {belanja}.')

df.to_csv('hasil_pelanggan_kmeans.csv', index=False)
evaluasi.to_csv('evaluasi_kmeans.csv', index=False)
print('Hasil disimpan ke hasil_pelanggan_kmeans.csv dan evaluasi_kmeans.csv.')


Klaster 0: 81 pelanggan, pendapatan sedang, dan skor belanja sedang.
Klaster 1: 39 pelanggan, pendapatan tinggi, dan skor belanja tinggi.
Klaster 2: 35 pelanggan, pendapatan tinggi, dan skor belanja rendah.
Klaster 3: 23 pelanggan, pendapatan rendah, dan skor belanja rendah.
Klaster 4: 22 pelanggan, pendapatan rendah, dan skor belanja tinggi.
Hasil disimpan ke hasil_pelanggan_kmeans.csv dan evaluasi_kmeans.csv.


## Kesimpulan
Dua fitur yang digunakan sudah memenuhi ketentuan tugas dan dapat membedakan pola pelanggan. Jumlah klaster dipilih melalui perbandingan Elbow dan nilai Silhouette. Ringkasan setiap klaster membantu melihat pelanggan yang pendapatan dan skor belanjanya rendah, sedang, atau tinggi.

Dari percobaan k = 2 sampai 10, k = 5 memiliki Silhouette tertinggi, yaitu sekitar 0,5547. Kelima kelompok menunjukkan pola pendapatan dan skor belanja yang berbeda, seperti pendapatan tinggi dengan belanja tinggi, serta pendapatan tinggi dengan belanja rendah.
